In [1]:
import pandas as pd

from datapreparation.data_preparation import Data_Preparation

path = "../mockdata/"

df_waterings = pd.read_csv(f'{path}waterings.csv')
df_sensors = pd.read_csv(f'{path}sensor_datas.csv')
df_plants = pd.read_csv(f'{path}plants.csv')

df_waterings

,PlantMAC,PredictedFutureWaterTime,LastWaterTime,WaterLevel,PumpTimeInSeconds
0,34:2c:d8:10:0f:2f,2025-01-12T21:31:57.620112,2025-01-01T12:00:00,86.69,25.9
1,34:2c:d8:10:0f:2f,2025-01-29T22:52:46.232448,2025-01-20T03:00:00,100.00,11.8
2,34:2c:d8:10:0f:2f,2025-02-04T10:59:59.314394,2025-01-27T12:00:00,100.00,11.3
3,34:2c:d8:10:0f:2f,2025-02-17T02:47:51.938046,2025-02-05T02:00:00,97.82,11.4
4,34:2c:d8:10:0f:2f,2025-02-28T03:56:49.372004,2025-02-16T07:00:00,93.86,10.9
...,...,...,...,...,...
435,72:da:2e:1f:34:14,2026-02-01T11:50:30.537914,2026-01-19T12:00:00,70.68,14.2
436,72:da:2e:1f:34:14,2026-02-07T15:16:19.368330,2026-01-26T12:00:00,45.91,15.3
437,72:da:2e:1f:34:14,2026-02-15T18:29:45.292930,2026-02-02T12:00:00,47.75,12.1
438,72:da:2e:1f:34:14,2026-02-22T16:00:45.531393,2026-02-09T12:00:00,60.08,12.5


In [2]:
mac_map = {mac: i for i, mac in enumerate(df_plants['MAC'].unique())}

df_waterings = df_waterings.drop(columns=['PredictedFutureWaterTime'])
df_waterings = df_waterings.rename(columns={'LastWaterTime': 'Timestamp'})

df_sensors['Timestamp']   = pd.to_datetime(df_sensors['Timestamp'])
df_waterings['Timestamp'] = pd.to_datetime(df_waterings['Timestamp'], format='mixed')

df_sensors['mac_id']   = df_sensors['PlantMAC'].map(mac_map)
df_waterings['mac_id'] = df_waterings['PlantMAC'].map(mac_map)
df_plants['mac_id']    = df_plants['MAC'].map(mac_map)

df_plants_cleaned = df_plants.drop(columns=['MAC', 'Username', 'Name'])

sensors_sorted   = df_sensors.sort_values('Timestamp').reset_index(drop=True).drop(columns=['PlantMAC'])
waterings_sorted = df_waterings.sort_values('Timestamp').reset_index(drop=True).drop(columns=['PlantMAC'])

df_water_sensors = pd.merge_asof(
    sensors_sorted,
    waterings_sorted[['mac_id', 'Timestamp', 'PumpTimeInSeconds','WaterLevel']].rename(columns={'Timestamp': 'LastWaterTimestamp'}),
    left_on='Timestamp',
    right_on='LastWaterTimestamp',
    by='mac_id',
    direction='backward'
)

df_water_sensors['seconds_since_watering'] = (
    df_water_sensors['Timestamp'] - df_water_sensors['LastWaterTimestamp']
).dt.total_seconds()

df_water_sensors = df_water_sensors.drop(columns=['LastWaterTimestamp', 'Timestamp'])

# Bring in plant optimal values
df_water_sensors = df_water_sensors.merge(df_plants_cleaned, on='mac_id', how='left')


df_water_sensors

,Temperature,AirHumidity,SoilHumidity,LightIntensity,mac_id,PumpTimeInSeconds,WaterLevel,seconds_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,20.19,52.35,57.88,1422.88,0,25.9,86.69,0.0,20.7,50.8,57.5,1282.9
1,18.78,52.77,58.26,1373.85,12,26.1,90.77,0.0,20.1,50.4,58.0,1282.1
2,18.38,68.59,72.08,1251.32,10,47.2,88.17,0.0,19.1,68.9,71.5,955.6
3,20.63,73.84,66.85,883.62,5,36.0,91.25,0.0,21.6,70.9,66.7,782.0
4,21.71,61.93,57.97,1248.57,6,25.9,96.48,0.0,22.1,59.3,57.5,793.8
...,...,...,...,...,...,...,...,...,...,...,...,...
149995,17.55,67.87,70.74,4.00,1,46.9,88.41,35996400.0,20.0,70.3,71.1,1047.1
149996,18.11,54.31,45.52,9.74,12,24.1,54.88,255600.0,20.1,50.4,58.0,1282.1
149997,18.98,52.22,45.00,3.85,0,11.0,69.89,151200.0,20.7,50.8,57.5,1282.9
149998,20.54,48.36,35.24,0.00,9,13.2,76.60,3600.0,22.3,50.2,50.6,722.5


In [3]:
for mac, group in df_water_sensors.groupby('mac_id'):
    pass

X = df_water_sensors.drop(columns=['mac_id', 'PumpTimeInSeconds'])
y = df_water_sensors['PumpTimeInSeconds']
X

,Temperature,AirHumidity,SoilHumidity,LightIntensity,WaterLevel,seconds_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,20.19,52.35,57.88,1422.88,86.69,0.0,20.7,50.8,57.5,1282.9
1,18.78,52.77,58.26,1373.85,90.77,0.0,20.1,50.4,58.0,1282.1
2,18.38,68.59,72.08,1251.32,88.17,0.0,19.1,68.9,71.5,955.6
3,20.63,73.84,66.85,883.62,91.25,0.0,21.6,70.9,66.7,782.0
4,21.71,61.93,57.97,1248.57,96.48,0.0,22.1,59.3,57.5,793.8
...,...,...,...,...,...,...,...,...,...,...
149995,17.55,67.87,70.74,4.00,88.41,35996400.0,20.0,70.3,71.1,1047.1
149996,18.11,54.31,45.52,9.74,54.88,255600.0,20.1,50.4,58.0,1282.1
149997,18.98,52.22,45.00,3.85,69.89,151200.0,20.7,50.8,57.5,1282.9
149998,20.54,48.36,35.24,0.00,76.60,3600.0,22.3,50.2,50.6,722.5
